# Preparación de datos

**Objetivos**
Detección de duplicados, valores imposibles y valores atípicos. Diagnóstico de huecos y regularidad temporal. Estandarización de formatos. Datos originales en raw/, datos formateados para ser usados en los siguientes NB en processed/.

**Datasets utilizados** Perfiles de hidrografía (freq. mensual), datos botellas de nutrientes y clorofila (freq. mensual) y datos simulados (freq. diaria) en la estacion E2GI.

In [9]:
## Este NB está en R porque me resulta más fácil para formatear datos,
## pero no hace falta reproducirlo, todos los archivos formateados
## que usarán el resto de NB están en data/processed/
library(lubridate)


Adjuntando el paquete: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union




**Dataset hidrografía**: capturados con CTD y sensores acoplados a la roseta. Tras el procesamiento, se generan datos para cada mes en forma de perfiles verticales con resolución espacial de 1m. En este ejemplo, estos son los datos de temperatura, salinidad, fluorescencia y oxígeno disuelto (formateados como *data.frame pandas* con resolucion mensual para **NB03**).

In [1]:
# Select hidro data
datos<-read.csv("../data/raw/Hidro_E2GI_insitu.csv")
#dim(datos)
#colnames(datos)
hidro_surface<-datos[datos$Press==5,]
#colnames(hidro_surface)
# Excluir concentraciones ausentes del cálculo de la media
validos <- !is.na(hidro_surface$Fluor)
datos <- hidro_surface[validos, ]

In [2]:
# Crear una fecha correspondiente al inicio de cada mes y agregar por mes
datos$mes <- as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",rep(1, length(datos$DD))), sep="-")))
meses <- sort(unique(datos$mes))
hidro_m <- data.frame(mes = meses,temp = rep(NA,length(meses)), sal = rep(NA,length(meses)),
                      fluor = rep(NA,length(meses)), o2 = rep(NA,length(meses)), n_muestras = integer(length(meses)))

for (i in seq_along(meses)) {
  valores <- datos$Temp[datos$mes == meses[i]]
  hidro_m$temp[i] <- mean(valores)
  valores <- datos$Sal[datos$mes == meses[i]]
  hidro_m$sal[i] <- mean(valores)
  valores <- datos$Fluor[datos$mes == meses[i]]
  hidro_m$fluor[i] <- mean(valores)
  valores <- datos$O2[datos$mes == meses[i]]
  hidro_m$o2[i] <- mean(valores)
  hidro_m$n_muestras[i] <- length(valores)
}

head(hidro_m, 2)

,mes,temp,sal,fluor,o2,n_muestras
,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,2001-04-01,13.5672,34.8607,0.133340,NA,1
2,2001-06-01,16.2020,35.1021,0.058948,NA,1


In [3]:
# Incorporar los meses sin observaciones desde un calendario mensual completo
calendario <- seq(from = min(hidro_m$mes),to = max(hidro_m$mes),by = "month")
indice <- match(calendario, hidro_m$mes)

df <- data.frame(
  mes = calendario,
  temp = hidro_m$temp[indice],
  sal = hidro_m$sal[indice],
  fluor = hidro_m$fluor[indice],
  o2 = hidro_m$o2[indice],
  n_muestras = hidro_m$n_muestras[indice])

# Resumen de cobertura
cat("Meses totales:", nrow(df), "\n")
cat("Meses con observaciones:", sum(!is.na(df$chla)), "\n")
cat("Meses sin observaciones:", sum(is.na(df$chla)), "\n")

cat(
  "Porcentaje de meses observados:",
  round(100 * mean(!is.na(df$chla)), 1),"%\n")

Meses totales: 256 
Meses con observaciones: 0 
Meses sin observaciones: 0 
Porcentaje de meses observados: NaN %


In [4]:
# Guardar df como CSV (los NB python lo leerán como df pandas)
write.csv(df,
  "../data/processed/Hidro_E2GI_surface_monthly.csv",
  row.names = FALSE,na = "")
colnames(df)

[1] "mes"        "temp"       "sal"        "fluor"      "o2"        
[6] "n_muestras"

**Datasets botellas**: medidos en el agua recogida con botellas acopladas a la roseta oceanográfica. Cada mes, forman perfiles con dato a profundidades discretas. En este ejemplo, estos son los datos de nutrientes y clorofila (formateados como *data.frame pandas*  con resolucion mensual para **NB03**).

In [5]:
datos<-read.csv("../data/raw/Chla_E2GI_insitu.csv")
#dim(datos)
#colnames(datos)
chla_surface<-datos[datos$Depth==0 & datos$Fraction=="GFF",]
#colnames(chla_surface)

# Excluir concentraciones ausentes del cálculo de la media
validos <- !is.na(chla_surface$Chl.a)
datos <- chla_surface[validos, ]

In [6]:
# Crear una fecha correspondiente al inicio de cada mes y agregar por mes
datos$mes <- as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",rep(1, length(datos$DD))), sep="-")))
meses <- sort(unique(datos$mes))
chla_m <- data.frame(mes = meses,chla = NA_real_,n_muestras = integer(length(meses)))

for (i in seq_along(meses)) {
  valores <- datos$Chl.a[datos$mes == meses[i]]
  chla_m$chla[i] <- mean(valores)
  chla_m$n_muestras[i] <- length(valores)
}

head(chla_m, 2)

,mes,chla,n_muestras
,<date>,<dbl>,<int>
1,2001-05-01,0.801,1
2,2001-06-01,0.211,1


In [7]:
# Incorporar los meses sin observaciones desde un calendario mensual completo
calendario <- seq(from = min(chla_m$mes),to = max(chla_m$mes),by = "month")
indice <- match(calendario, chla_m$mes)

df <- data.frame(
  mes = calendario,
  chla = chla_m$chla[indice],
  n_muestras = chla_m$n_muestras[indice])

# Resumen de cobertura
cat("Meses totales:", nrow(df), "\n")
cat("Meses con observaciones:", sum(!is.na(df$chla)), "\n")
cat("Meses sin observaciones:", sum(is.na(df$chla)), "\n")

cat("Porcentaje de meses observados:",
  round(100 * mean(!is.na(df$chla)), 1),"%\n")

Meses totales: 260 
Meses con observaciones: 169 
Meses sin observaciones: 91 
Porcentaje de meses observados: 65 %


In [8]:
# Guardar df como CSV
write.csv(df,"../data/processed/Chla_E2GI_surface_monthly.csv",
  row.names = FALSE,na = "")

**Datasets hidrografía**: en este ejemplo, los datos de temperatura y salinidad se han formateado como perfiles leíbles por GOTM-FABM para ser usados como condición inicial y/o nudging en **NB05** (.txt).

In [ ]:
#### Hidro RADIALES   2001-2022         
datos<-read.csv("../data/raw/Hidro_E2GI_insitu.csv.csv")
#colnames(datos)

fechas<-as.Date(paste(paste(datos$YYYY,sprintf("%02d",datos$MM),sprintf("%02d",datos$DD), sep="-")))
julianos<- julian(fechas, origin=as.Date("2001-01-01"))
muestreos<-unique(fechas)

# Resumen de cobertura
cat("Rango fechas muestreos:", range fechas, "\n")
cat("Meses con observaciones:", length(muestreos), "\n")

In [ ]:
## Temperature           
valor1<-datos$Temp
profun<-datos$Press
path_observa<-"../data/processed/"
sink(paste(path_observa,"temp_E2GI.txt",sep=""))

for (k in c(1:length(muestreos))){
  valo<-valor1[fechas==muestreos[k]]
  prof<-profun[fechas==muestreos[k]]
  #plot(valo,prof,ylim=c(100,0))
  # remove NAs
  prof<-prof[!is.na(valo)]
  valo<-valo[!is.na(valo)]
  fechita<-paste(muestreos[k],"12:00:00") # refine getting HH:MM:SS
  # write profile
  header <- paste(format(fechita, width = 10, justify = "left"),length(prof),"2", sep="\t")
  datoss <- cbind(paste("",format(-prof,    width = 6), sep=""),paste("",format(valo,  width = 8, justify = "right"), sep=""))
  cat(paste(header,"\n", sep=""), append = T,file=paste(path_observa,"temp_E2GI.txt",sep=""))
  write.table(datoss, file=paste(path_observa,"temp_E2GI.txt",sep=""),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE, append=TRUE)}
sink()

In [ ]:
## Salinity
valor1<-datos$Sal
profun<-datos$Press
path_observa<-"../data/processed/"
sink(paste(path_observa,"sal_E2GI.txt",sep=""))

for (k in c(1:length(muestreos))){
  valo<-valor1[fechas==muestreos[k]]
  prof<-profun[fechas==muestreos[k]]
  #plot(valo,prof,ylim=c(100,0))
  # remove NAs
  prof<-prof[!is.na(valo)]
  valo<-valo[!is.na(valo)]
  fechita<-paste(muestreos[k],"12:00:00")
  # write profile
  header <- paste(format(fechita, width = 10, justify = "left"),length(prof),"2", sep="\t")
  datoss <- cbind(paste("",format(-prof,    width = 6), sep=""),paste("",format(valo,  width = 8, justify = "right"), sep=""))
  cat(paste(header,"\n", sep=""), append = T,file=paste(path_observa,"sal_E2GI.txt",sep=""))
  write.table(datoss, file=paste(path_observa,"sal_E2GI.txt",sep=""),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE, append=TRUE)}
sink() 

**Datasets BGC**: en este ejemplo, los datos de clorofila (botellas) se han formateado como observaciones leíbles por GOTM-FABM para ser usados en calibración y/o asimilación en **NB07**(.obs). *pendiente* Ampliar con datos de fluorescencia y oxígeno (hidro) y nutrientes (botellas).

In [10]:
#### Chla Radiales 2001-2022
datos<-read.csv("../data/raw/Chla_E2GI_insitu.csv")
colnames(datos)

[1] "Date"     "YYYY"     "MM"       "DD"       "Station"  "Fraction" "Depth"   
[8] "Chl.a"

In [11]:
## CHLA total GFF          
#unique(datos$Station)
#unique(datos$Fraction)
TCHLA<-datos[datos$Station=="E2GI" & datos$Fraction=="GFF",]

FECH<-paste(paste(TCHLA$YYYY,    
                  sprintf("%02d",TCHLA$MM),   
                  sprintf("%02d",TCHLA$DD), sep="-"),
            paste(sprintf("%02d",12),
                  sprintf("%02d",0),
                  "00", sep=":"))
PROF<-TCHLA$Depth
VALO<-TCHLA$Chl.a
# Remove NAs
FECH<-FECH[!is.na(VALO)]
PROF<-PROF[!is.na(VALO)]
VALO<-VALO[!is.na(VALO)]

# Resumen de cobertura
cat("Numero valores:", length(VALO), "\n")

## save .obs file
path_observa<-"../data/processed/"
sink(paste(path_observa,"TChla_E2GI.obs",sep=""))
datoss  <- cbind(paste("",format(FECH, width = 10, justify = "left"), sep=""),
                 paste("",format(-PROF, width = 5, justify = "left"), sep=""),
                 paste("",format(VALO,  width = 5, justify = "right"), sep=""))
write.table(datoss, file=paste(path_observa,"TChla_E2GI.obs",sep=""),
            sep = "  ", row.names = FALSE, col.names = FALSE, quote = FALSE, append=F)
sink()

Numero valores: 1357 


In [ ]:
## 0.2um < Chlorophyll < 2um (picofitoplancton)
TCHLA<-datos[datos$Station=="E2GI" & datos$Fraction=="0.2_um",]

FECH<-paste(paste(TCHLA$YYYY,    
                  sprintf("%02d",TCHLA$MM),   
                  sprintf("%02d",TCHLA$DD), sep="-"),
            paste(sprintf("%02d",12),
                  sprintf("%02d",0),
                  "00", sep=":"))
PROF<-TCHLA$Depth
VALO<-TCHLA$Chl.a

# Remove NAs
FECH<-FECH[!is.na(VALO)]
PROF<-PROF[!is.na(VALO)]
VALO<-VALO[!is.na(VALO)]

# Resumen de cobertura
cat("Numero valores:", length(VALO), "\n")

## save .obs file
path_observa<-"../data/processed/"
sink(paste(path_observa,"pChla_E2GI.obs",sep=""))
datoss  <- cbind(paste("",format(FECH, width = 10, justify = "left"), sep=""),
                 paste("",format(-PROF, width = 5, justify = "left"), sep=""),
                 paste("",format(VALO,  width = 5, justify = "right"), sep=""))
write.table(datoss, file=paste(path_observa,"pChla_E2GI.obs",sep=""),
            sep = "  ", row.names = FALSE, col.names = FALSE, quote = FALSE, append=F)
sink()

In [ ]:
## 2um < Chlorophyll < 20um (nanofitoplancton)
TCHLA<-datos[datos$Station=="E2GI" & datos$Fraction=="2_um",]

FECH<-paste(paste(TCHLA$YYYY,    
                  sprintf("%02d",TCHLA$MM),   
                  sprintf("%02d",TCHLA$DD), sep="-"),
            paste(sprintf("%02d",12),
                  sprintf("%02d",0),
                  "00", sep=":"))
PROF<-TCHLA$Depth
VALO<-TCHLA$Chl.a

# Remove NAs
FECH<-FECH[!is.na(VALO)]
PROF<-PROF[!is.na(VALO)]
VALO<-VALO[!is.na(VALO)]

# Resumen de cobertura
cat("Numero valores:", length(VALO), "\n")

## save .obs file
path_observa<-"../data/processed/"
sink(paste(path_observa,"nChla_E2GI.obs",sep=""))
datoss  <- cbind(paste("",format(FECH, width = 10, justify = "left"), sep=""),
                 paste("",format(-PROF, width = 5, justify = "left"), sep=""),
                 paste("",format(VALO,  width = 5, justify = "right"), sep=""))
write.table(datoss, file=paste(path_observa,"nChla_E2GI.obs",sep=""),
            sep = "  ", row.names = FALSE, col.names = FALSE, quote = FALSE, append=F)
sink()

In [ ]:
## 20um < Chlorophyll  (microfitoplancton)
TCHLA<-datos[datos$Station=="E2GI" & datos$Fraction=="20_um",]

FECH<-paste(paste(TCHLA$YYYY,    
                  sprintf("%02d",TCHLA$MM),   
                  sprintf("%02d",TCHLA$DD), sep="-"),
            paste(sprintf("%02d",12),
                  sprintf("%02d",0),
                  "00", sep=":"))
PROF<-TCHLA$Depth
VALO<-TCHLA$Chl.a

# Remove NAs
FECH<-FECH[!is.na(VALO)]
PROF<-PROF[!is.na(VALO)]
VALO<-VALO[!is.na(VALO)]

# Resumen de cobertura
cat("Numero valores:", length(VALO), "\n")

## save .obs file
path_observa<-"../data/processed/"
sink(paste(path_observa,"mChla_E2GI.obs",sep=""))
datoss  <- cbind(paste("",format(FECH, width = 10, justify = "left"), sep=""),
                 paste("",format(-PROF, width = 5, justify = "left"), sep=""),
                 paste("",format(VALO,  width = 5, justify = "right"), sep=""))
write.table(datoss, file=paste(path_observa,"mChla_E2GI.obs",sep=""),
            sep = "  ", row.names = FALSE, col.names = FALSE, quote = FALSE, append=F)
sink()

**Dataset satelite**: formatear una serie temporal de clorofila de satelite como observaciones leíbles por GOTM-FABM para ser usados en calibración y/o asimilación en **NB07** (.obs).

In [ ]:
## pendiente